In [1]:
import pandas as pd
from tqdm import tqdm
import numpy as np
import time

from __future__ import annotations
from yandex_cloud_ml_sdk import YCloudML

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option("display.max_colwidth", None)

In [6]:
data_test = pd.read_csv('data_new_staging_articles.csv')

data_test.head(2)

,id,account_id,contractor_id,date,payments_amount,purpose,article_id,expenditure,project_id,counterpartie_id,donor_id,robot_id,donor_cat_id,accounts__id,accounts__user_id,articles__id,articles__user_id,articles__parent_id,articles__name,projects__id,projects__user_id,projects__parent_id,projects__name,counterparties__id,counterparties__user_id,counterparties__parent_id,counterparties__name,robots__id,robots__user_id,article_alternative_names__user_id,target
0,1102148,2881,41747,2025-10-09,29.0,"//Приложение//кол-во распор. 1 //Перевод ср-в согл. реестра переводов за 08.10.2025 по Дог. 560/2017 от 07.12.2017, удерж. комис. ,00 руб. НДС не обл.",48528,incoming,3801,0,0,13320,9985,2881.0,962.0,48528.0,962.0,41226.0,мкб_терминал,3801.0,962.0,0.0,Уставные деятельность,9985.0,962.0,9982.0,...массовые,13320.0,962.0,NaN,1
1,1102255,114,75,2025-10-09,2605.0,Перевод денежных средств по договору присоединения к условиям оказания услуг ИТО при осуществлении переводов денежных средств благотворительным организациям от 15/04/2022. Без учета НДС.,49607,incoming,1980,0,0,-1,2092,114.0,187.0,49607.0,187.0,47651.0,вк добро физлица,1980.0,187.0,0.0,Уставные цели,2092.0,187.0,13621.0,ВК добро,NaN,NaN,NaN,2


In [7]:
# разворачиваем таргет в текстовые категории - для LLM
uc_codex = {"пожертвования от физических лиц (напрямую)":1,
        "пожертвования через платформы":2,
        "пожертвования от юридических лиц (напрямую)":3,
        "прочие недоходные операции":4,
        "продажа услуг":5,
        "продажа товаров":6,
        "финансовые доходы":7,
        "членские и учредительские взносы":8,
        "гранты субсидии конкурсы":9}

uc_codex_rev = {v: k for k, v in uc_codex.items()}

data_test['universal_category'] = data_test['target'].map(uc_codex_rev)

data_test.head(3)

,id,account_id,contractor_id,date,payments_amount,purpose,article_id,expenditure,project_id,counterpartie_id,donor_id,robot_id,donor_cat_id,accounts__id,accounts__user_id,articles__id,articles__user_id,articles__parent_id,articles__name,projects__id,projects__user_id,projects__parent_id,projects__name,counterparties__id,counterparties__user_id,counterparties__parent_id,counterparties__name,robots__id,robots__user_id,article_alternative_names__user_id,target,universal_category
0,1102148,2881,41747,2025-10-09,29.00,"//Приложение//кол-во распор. 1 //Перевод ср-в согл. реестра переводов за 08.10.2025 по Дог. 560/2017 от 07.12.2017, удерж. комис. ,00 руб. НДС не обл.",48528,incoming,3801,0,0,13320,9985,2881.0,962.0,48528.0,962.0,41226.0,мкб_терминал,3801.0,962.0,0.0,Уставные деятельность,9985.0,962.0,9982.0,...массовые,13320.0,962.0,NaN,1,пожертвования от физических лиц (напрямую)
1,1102255,114,75,2025-10-09,2605.00,Перевод денежных средств по договору присоединения к условиям оказания услуг ИТО при осуществлении переводов денежных средств благотворительным организациям от 15/04/2022. Без учета НДС.,49607,incoming,1980,0,0,-1,2092,114.0,187.0,49607.0,187.0,47651.0,вк добро физлица,1980.0,187.0,0.0,Уставные цели,2092.0,187.0,13621.0,ВК добро,NaN,NaN,NaN,2,пожертвования через платформы
2,1102256,114,46,2025-10-09,13736.01,"Перечисление денежных средств по Договору №ПД-1299 от 2024-01-30 по платежам за 2025-10-08. DS.2471242. Сумма удержанной комиссии 280,33 руб.",49853,incoming,1980,0,0,-1,12584,114.0,187.0,49853.0,187.0,47651.0,тбанк физлица,1980.0,187.0,0.0,Уставные цели,12584.0,187.0,13621.0,ТБанк,NaN,NaN,NaN,1,пожертвования от физических лиц (напрямую)


In [ ]:
data_test.loc[:, 'text']  = data_test.apply(
    lambda x: (
        f"""назначение платежа: {x['purpose']}
название статьи: {x['articles__name']}
название проекта: {x['projects__name']}
категория донора: {x['counterparties__name']}"""
    ),
    axis=1
)



In [14]:
y_pred_ygpt_ft = pd.DataFrame(columns=["universal_category"])

sdk = YCloudML(
        folder_id="YOUR_FOLDER_ID",
        auth="YOUR_YANDEX_CLOUD_TOKEN",
    )

model = sdk.models.text_classifiers(
        "cls://YOUR_FOLDER_ID/yandexgpt-lite/latest@tamros8p69qq7ribmm06k"
    )

for index__, text in tqdm(data_test['text'].items(), total=len(data_test)):
    attempts = 0
    max_attempts = 3
    while attempts < max_attempts:
        try:
            result = model.run(str(text))  # приводим текст к строке
            #print(result, "\n")
            best_label = max(result, key=lambda x: x.confidence).label
            y_pred_ygpt_ft.loc[index__, "universal_category"] = best_label
            break
        except Exception as e:
            attempts += 1
            print(f"Ошибка на index {index__}, попытка {attempts}/{max_attempts}: {e}")
            time.sleep(2)
    else:
        # если все попытки неудачные, присваиваем NaN
        y_pred_ygpt_ft.loc[index__, "universal_category"] = np.nan


100%|██████████| 1721/1721 [1:00:13<00:00,  2.10s/it]


In [15]:
y_pred_ygpt_ft.to_csv("y_pred_ygpt_ft.csv", index=True, header=["universal_category"])
data_test['universal_category'].to_csv("y_test.csv", index=True, header=["universal_category"])
data_test['text'].to_csv("X_test.csv", index=True, header=["text"])